In [6]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

os.environ['GROQ_API_KEY'] = "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"
load_dotenv(".env", override=True)

# Maak verbinding met het Groq taalmodel
# temperature=0.3 zorgt voor voorspelbare, consistente antwoorden (0=altijd hetzelfde, 1=creatief)
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.3)


In [7]:
# --- SCHEMA'S ---
# Pydantic BaseModel definieert de 'mal' waar de output van het model in moet passen.
# Het model wordt gedwongen altijd deze velden terug te geven, nooit meer of minder.

# Schema voor één bereidingsstap: een beschrijving + geschatte tijd
class StapSchema(BaseModel):
    stap: str = Field(description="Beschrijving van de stap")
    tijd: str = Field(description="Geschatte tijd voor deze stap")

# Hoofdschema voor een volledig recept
# 'stappen' is een lijst van StapSchema-objecten: je nestelt het ene schema in het andere
class RecipeSchema(BaseModel):
    beschrijving: str = Field(description="Beschrijving van recept")
    ingredienten: list[str] = Field(description="Lijst van ingredienten uit het recept")
    stappen: list[StapSchema] = Field(description="Stappen in het recept met geschatte tijd")


In [8]:
# --- PROMPT TEMPLATE ---
# Gino d'Acampo persona: Italiaans, passievol, trots op zijn Nona,
# geen geduld voor twijfelaars, maar hartelijk en enthousiast
prompt = ChatPromptTemplate.from_messages([
    ("human", """### Chi sei (Wie ben je)
Jij bent Gino d'Acampo, de meest gepassioneerde Italiaanse chef van de wereld.
Je kookt zoals je Nona je heeft geleerd in Napels, en NIEMAND, absoluut niemand 
twijfelt aan de Italiaanse keuken van jouw Nona. Wie dat toch doet krijgt een
vermanende blik en de zin: "My Nona would never forgive you for that."

Je schrijft stappen altijd:
- In de jij-vorm, kort en actief ('Snijd de ui', 'Voeg de pasta toe')
- Met een vleugje Italiaans enthousiasme ('Bellissimo!', 'Perfetto!', 'Magnifico!')
- Beginnersvriendelijk, geen vakjargon, alsof je het aan je beste vriend uitlegt
- Met een korte aanmoediging bij moeilijkere stappen
- Nooit met shortcuts die je Nona zou afkeuren (geen potjes saus, geen zakjes)

### Il Ricettario (Het Recept)
{recept}

### Il Compito (De Taak)
Analyseer het recept en transformeer het naar SousChef-stijl à la Gino:
1. Schrijf een beschrijving vol passie — vertel waarom dit gerecht geweldig is
2. Maak een volledige ingrediëntenlijst
3. Schrijf elke stap beginnersvriendelijk, in Gino's stem, met tijdschatting
4. Voeg bij minstens één stap een Gino-opmerking toe zoals:
   'My Nona always said: ...' of 'In Italia doen we dit ZO, niet anders!'

### Le Regole di Nona (De Regels)
- Schrijf alsof je live kookt op televisie
- Wees enthousiast maar duidelijk
- Goedkope shortcuts zijn VERBODEN — Nona kijkt mee
- Gebruik maximaal 1-2 Italiaanse uitroepen per recept (niet overdrijven)

### Output Format (JSON)
{{
  "beschrijving": "passionele beschrijving van het gerecht",
  "ingredienten": ["ingredient 1", "ingredient 2", "ingredient 3"],
  "stappen": [
    {{"stap": "beginnersvriendelijke instructie in Gino's stem", "tijd": "x minuten"}}
  ]
}}
""")
])

print("✅ Gino d'Acampo prompt geladen — Nona is trots!")


✅ Gino d'Acampo prompt geladen — Nona is trots!


In [9]:
# --- EVALUATIESET: 10 GEVARIEERDE RECEPTEN ---
# Gevarieerd in: taal (NL/EN), formaat (gestructureerd/ongestructureerd),
# complexiteit (simpel/meerdere componenten) en type gerecht

# 1. Nederlands, gestructureerd, simpel
recept_1 = """One-pan Chicken and Garlic Rice
Total Time: 35 minuten | Porties: 4
Ingrediënten: 5 kipfilets, 1 ui, 2 teentjes knoflook, 1 kop rijst, 1.5 kop kippenbouillon,
1 tl paprikapoeder, 1 tl tijm, zout, peper, olijfolie
Bereidingswijze:
Kruid de kip met paprika, tijm, zout en peper.
Bak de kip in olijfolie 4 min per kant. Leg apart.
Fruit ui en knoflook in dezelfde pan.
Voeg rijst toe en roer 1 minuut.
Voeg bouillon toe, leg kip erop en dek af.
Kook 20 minuten op laag vuur tot rijst gaar is."""

# 2. Engels, gestructureerd, simpel
recept_2 = """Creamy Tomato Gnocchi
Total Time: 20 min | Serves: 4
Ingredients: 500g gnocchi, 400g cherry tomatoes, 1 onion, 3 garlic cloves,
200ml cream, 50g parmesan, 2 tbsp butter, pepper, fresh basil
Instructions:
Chop onion, mince garlic, halve tomatoes.
Pan-fry gnocchi in butter until golden, about 8 minutes.
Sauté onion and garlic 2 minutes, add tomatoes.
Stir in cream and parmesan until creamy.
Toss gnocchi back in, season with pepper and basil."""

# 3. Nederlands, ongestructureerd (lopende tekst), gemiddeld
recept_3 = """Erwtensoep
Dit is oma's klassieke erwtensoep. Je hebt nodig: 500g spliterwten, 1 rookworst,
2 aardappelen, 2 wortels, 1 selderijknol, 1 ui, zout en peper.
Week de erwten een nacht. Kook ze daarna in 1.5 liter water met de ui gedurende
45 minuten. Voeg dan de blokjes aardappel, wortel en selderij toe en kook nog
20 minuten. Snij de rookworst in plakjes en voeg toe. Breng op smaak met zout
en peper. Lekker met roggebrood."""

# 4. Engels, ongestructureerd (blog-stijl), simpel
recept_4 = """The Best Avocado Toast
Okay so I make this literally every morning and it takes like 5 minutes.
You need 2 slices of sourdough bread, 1 ripe avocado, a lemon, some red pepper
flakes, salt, and if you're feeling fancy, a poached egg on top.
Toast your bread. While that's happening mash your avocado with a squeeze of
lemon and a pinch of salt. Spread it on the toast, sprinkle the chili flakes,
add the egg if you want. Done!"""

# 5. Nederlands, meerdere componenten, complex
recept_5 = """Nasi Goreng met Gebakken Ei
Bereidingstijd: 30 min | Porties: 2
Voor de nasi: 300g gekookte rijst (dag oud), 2 eieren, 1 ui, 2 teentjes knoflook,
1 tl sambal, 2 el ketjap manis, 1 tl trassi, zonnebloemolie
Voor het gebakken ei: 2 eieren, olie, zout
Bereidingswijze nasi:
Fruit ui en knoflook in olie. Voeg trassi toe en bak 1 minuut.
Voeg rijst toe en roerbak 3 minuten op hoog vuur.
Maak ruimte in de pan, breek de eieren erin en roer door de rijst.
Voeg sambal en ketjap toe, roerbak nog 2 minuten.
Bereidingswijze gebakken ei:
Bak de eieren in een aparte pan in hete olie, bestrooi met zout.
Serveer de nasi met het gebakken ei erop."""

# 6. Engels, tabel-formaat (ingrediënten als lijst met hoeveelheden)
recept_6 = """Classic Spaghetti Bolognese
Time: 45 min | Serves: 4
INGREDIENTS
400g | spaghetti
500g | minced beef
1 | onion, diced
2 cloves | garlic, minced
400g | canned tomatoes
2 tbsp | tomato paste
1 tsp | dried oregano
1 glass | red wine
salt, pepper, olive oil
INSTRUCTIONS
1. Brown the mince in olive oil, set aside.
2. Soften onion and garlic in the same pan.
3. Add tomato paste and oregano, cook 1 min.
4. Pour in wine, let it reduce by half.
5. Add canned tomatoes and browned mince, simmer 25 min.
6. Cook spaghetti al dente, serve with sauce."""

# 7. Nederlands, minimale opmaak, snel recept
recept_7 = """Roerei met spinazie
4 eieren, handvol spinazie, 1 teentje knoflook, boter, zout, peper, nootmuskaat.
Smelt boter in pan. Bak knoflook kort aan. Voeg spinazie toe tot geslonken.
Klop eieren los met zout en peper. Giet over spinazie.
Roer zachtjes tot eieren net gestold zijn. Rasp wat nootmuskaat erover."""

# 8. Engels, bakrecept, precies en technisch
recept_8 = """Banana Bread
Prep: 15 min | Bake: 60 min | Makes: 1 loaf
Ingredients: 3 very ripe bananas, 80g melted butter, 150g sugar, 1 egg (beaten),
1 tsp vanilla, 1 tsp baking soda, pinch of salt, 190g all-purpose flour
Method:
Preheat oven to 175°C. Grease a loaf pan.
Mash bananas in a bowl until smooth.
Mix in melted butter.
Stir in sugar, egg, and vanilla.
Add baking soda and salt, mix well.
Fold in flour until just combined — do not overmix.
Pour into pan and bake 60 minutes until a skewer comes out clean.
Cool 10 minutes before slicing."""

# 9. Nederlands, vegetarisch, moderne opmaak
recept_9 = """Thaise Groene Curry met Kikkererwten
⏱ 25 minuten | 🍽 2 personen | 🌱 Vegan
Wat heb je nodig?
- 1 blik kikkererwten (400g), uitgelekt
- 1 blik kokosmelk (400ml)
- 2 el groene currypasta
- 1 courgette, in blokjes
- 1 rode paprika, in reepjes
- 1 el vissaus (of sojasaus voor vegan)
- 1 tl suiker
- verse koriander, limoensap
- gestoomde rijst om te serveren
Hoe maak je het?
Bak de currypasta 2 minuten in een droge wok.
Voeg kokosmelk toe en breng aan de kook.
Voeg groenten en kikkererwten toe, kook 10 minuten.
Breng op smaak met vissaus en suiker.
Serveer met rijst, koriander en limoensap."""

# 10. Engels, heel minimaal, bijna geen structuur
recept_10 = """Quick Stir Fry
chicken breast, broccoli, soy sauce, garlic, ginger, sesame oil, cornstarch, rice
slice chicken thin, marinate in soy sauce and cornstarch 10 mins
fry garlic and ginger in hot oil, add chicken cook through
add broccoli, splash of soy sauce, stir fry 3 mins
drizzle sesame oil, serve over rice"""

# Alle recepten in één lijst voor de evaluatie
evaluatieset = [
    ("One-pan Chicken and Garlic Rice", recept_1),
    ("Creamy Tomato Gnocchi",           recept_2),
    ("Erwtensoep",                       recept_3),
    ("Avocado Toast",                    recept_4),
    ("Nasi Goreng",                      recept_5),
    ("Spaghetti Bolognese",              recept_6),
    ("Roerei met Spinazie",              recept_7),
    ("Banana Bread",                     recept_8),
    ("Thaise Groene Curry",              recept_9),
    ("Quick Stir Fry",                   recept_10),
]

print(f"✅ Evaluatieset klaar: {len(evaluatieset)} recepten geladen")


✅ Evaluatieset klaar: 10 recepten geladen


In [10]:
# ============================================================
# LANGGRAPH WORKFLOW
# ============================================================
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional
import json, csv

# STATE
class ReceptState(TypedDict):
    recept_tekst: str
    recept_schema: Optional[dict]
    validatie_fouten: list
    reparatie_pogingen: int
    export_pad: Optional[str]

# NODE 1: PARSE
def parse_recept(state: ReceptState) -> dict:
    print("  ▶ Node 1: Parsen...")
    parse_prompt = ChatPromptTemplate.from_messages([
        ("human", """Je bent SousChef, een vriendelijke kookassistent die recepten omzet
naar heldere, beginnersvriendelijke instructies. Gebruik korte actieve zinnen.
Geen vakjargon. Moedig aan waar mogelijk (bijv. 'Dat ruikt al geweldig!').

Recepttekst:
{recept}

Geef ALLEEN een JSON object terug:
{{
  "beschrijving": "korte beschrijving van het gerecht",
  "ingredienten": ["ingredient 1", "ingredient 2"],
  "stappen": [{{"stap": "wat te doen", "tijd": "x minuten"}}]
}}
""")
    ])
    structured_llm = llm.with_structured_output(RecipeSchema, method="json_mode")
    chain = parse_prompt | structured_llm
    response = chain.invoke({"recept": state["recept_tekst"]})
    return {"recept_schema": response.model_dump()}

# NODE 2: VALIDEER
def valideer_recept(state: ReceptState) -> dict:
    print("  ▶ Node 2: Valideren...")
    fouten = []
    schema = state["recept_schema"]
    if not schema.get("beschrijving") or len(schema["beschrijving"]) < 10:
        fouten.append("Beschrijving te kort")
    if not schema.get("ingredienten") or len(schema["ingredienten"]) < 2:
        fouten.append("Minder dan 2 ingrediënten")
    if not schema.get("stappen") or len(schema["stappen"]) < 2:
        fouten.append("Minder dan 2 stappen")
    for i, stap in enumerate(schema.get("stappen", [])):
        if not stap.get("tijd"):
            fouten.append(f"Stap {i+1} mist tijdschatting")
    if fouten:
        print(f"  ⚠ Fouten: {fouten}")
    else:
        print("  ✓ Validatie geslaagd")
    return {"validatie_fouten": fouten}

# CONDITIE
def bepaal_volgende_stap(state: ReceptState) -> str:
    if state["validatie_fouten"] and state["reparatie_pogingen"] < 2:
        return "repareer"
    return "exporteer"

# NODE 3: REPAREER
def repareer_recept(state: ReceptState) -> dict:
    pogingen = state["reparatie_pogingen"] + 1
    print(f"  ▶ Node 3: Repareren (poging {pogingen})...")
    repareer_prompt = ChatPromptTemplate.from_messages([
        ("human", """Het recept-schema heeft fouten. Verbeter het als SousChef.
Huidig schema: {schema}
Fouten: {fouten}
Originele tekst: {recept_tekst}
Geef verbeterd JSON terug:
{{
  "beschrijving": "...",
  "ingredienten": ["..."],
  "stappen": [{{"stap": "...", "tijd": "..."}}]
}}
""")
    ])
    structured_llm = llm.with_structured_output(RecipeSchema, method="json_mode")
    chain = repareer_prompt | structured_llm
    response = chain.invoke({
        "schema": json.dumps(state["recept_schema"], ensure_ascii=False),
        "fouten": "\n".join(state["validatie_fouten"]),
        "recept_tekst": state["recept_tekst"]
    })
    return {"recept_schema": response.model_dump(), "reparatie_pogingen": pogingen, "validatie_fouten": []}

# NODE 4: EXPORTEER (JSON per recept)
def exporteer_recept(state: ReceptState) -> dict:
    print("  ▶ Node 4: Exporteren...")
    naam = state["recept_schema"]["beschrijving"][:30].replace(" ", "_").replace("/", "-")
    pad = f"recept_{naam}.json"
    with open(pad, "w", encoding="utf-8") as f:
        json.dump(state["recept_schema"], f, indent=2, ensure_ascii=False)
    return {"export_pad": pad}

# GRAPH BOUWEN
workflow = StateGraph(ReceptState)
workflow.add_node("parse", parse_recept)
workflow.add_node("valideer", valideer_recept)
workflow.add_node("repareer", repareer_recept)
workflow.add_node("exporteer", exporteer_recept)
workflow.add_edge(START, "parse")
workflow.add_edge("parse", "valideer")
workflow.add_conditional_edges("valideer", bepaal_volgende_stap, {"repareer": "repareer", "exporteer": "exporteer"})
workflow.add_edge("repareer", "valideer")
workflow.add_edge("exporteer", END)
app = workflow.compile()
print("✅ LangGraph workflow klaar!\n")

# ============================================================
# UITVOEREN OP EVALUATIESET
# ============================================================
recepten_output = []

for naam, recept in evaluatieset:
    print(f"{'='*50}")
    print(f"Verwerken: {naam}")
    resultaat = app.invoke({
        "recept_tekst": recept,
        "recept_schema": None,
        "validatie_fouten": [],
        "reparatie_pogingen": 0,
        "export_pad": None
    })
    # Voeg de naam toe aan het schema voor de CSV
    schema = resultaat["recept_schema"]
    schema["naam"] = naam
    schema["reparatie_pogingen"] = resultaat["reparatie_pogingen"]
    recepten_output.append(schema)
    print(f"  ✓ Klaar ({len(schema['stappen'])} stappen, {len(schema['ingredienten'])} ingrediënten)")

# ============================================================
# CSV EXPORT (spreadsheet-ready)
# ============================================================
# Elke rij = één recept
# Stappen en ingrediënten worden samengevoegd als tekst in één cel
csv_pad = "souschef_evaluatie.csv"

with open(csv_pad, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    # Header
    writer.writerow(["naam", "beschrijving", "aantal_ingredienten", "ingredienten",
                     "aantal_stappen", "stappen", "reparatie_pogingen"])
    for r in recepten_output:
        # Stappen samenvoegen als genummerde lijst
        stappen_tekst = " | ".join([f"{i+1}. {s['stap']} ({s['tijd']})" for i, s in enumerate(r["stappen"])])
        ingredienten_tekst = " | ".join(r["ingredienten"])
        writer.writerow([
            r["naam"],
            r["beschrijving"],
            len(r["ingredienten"]),
            ingredienten_tekst,
            len(r["stappen"]),
            stappen_tekst,
            r["reparatie_pogingen"]
        ])

# ============================================================
# JSON EXPORT (API-ready, alle recepten in één bestand)
# ============================================================
json_pad = "souschef_evaluatie.json"
with open(json_pad, "w", encoding="utf-8") as f:
    json.dump(recepten_output, f, indent=2, ensure_ascii=False)

print(f"\n{'='*50}")
print(f"✅ Evaluatie compleet!")
print(f"   {len(recepten_output)} recepten verwerkt")
print(f"   📄 CSV:  {csv_pad}")
print(f"   📦 JSON: {json_pad}")


✅ LangGraph workflow klaar!

Verwerken: One-pan Chicken and Garlic Rice
  ▶ Node 1: Parsen...
  ▶ Node 2: Valideren...
  ✓ Validatie geslaagd
  ▶ Node 4: Exporteren...
  ✓ Klaar (6 stappen, 10 ingrediënten)
Verwerken: Creamy Tomato Gnocchi
  ▶ Node 1: Parsen...
  ▶ Node 2: Valideren...
  ✓ Validatie geslaagd
  ▶ Node 4: Exporteren...
  ✓ Klaar (5 stappen, 9 ingrediënten)
Verwerken: Erwtensoep
  ▶ Node 1: Parsen...
  ▶ Node 2: Valideren...
  ⚠ Fouten: ['Stap 1 mist tijdschatting', 'Stap 4 mist tijdschatting', 'Stap 5 mist tijdschatting']
  ▶ Node 3: Repareren (poging 1)...
  ▶ Node 2: Valideren...
  ⚠ Fouten: ['Stap 1 mist tijdschatting', 'Stap 4 mist tijdschatting', 'Stap 5 mist tijdschatting']
  ▶ Node 3: Repareren (poging 2)...
  ▶ Node 2: Valideren...
  ✓ Validatie geslaagd
  ▶ Node 4: Exporteren...
  ✓ Klaar (5 stappen, 8 ingrediënten)
Verwerken: Avocado Toast
  ▶ Node 1: Parsen...
  ▶ Node 2: Valideren...
  ✓ Validatie geslaagd
  ▶ Node 4: Exporteren...
  ✓ Klaar (5 stappen, 6 ing